## Libraries

In [1]:
!pip install -q \
  datasets \
  transformers \
  peft \
  huggingface_hub \
  ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import sys
import time
import torch
import ipywidgets
from torch.utils.data import Dataset
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from itertools import islice
from huggingface_hub import login
from huggingface_hub import HfFolder
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:32" # Allows to split memory into pieces when allocating (normally the GPU VRAM needs a contiguous block of memory). Caps memory block size to maximum of 32 MB

## Load model & dataset

##### Login to huggingface

In [3]:
login()

#### Load base model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model     = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype="auto")

#### Load fine-tuned model

In [4]:
tokenizer = AutoTokenizer.from_pretrained("eduhuemar001/tinyllama-german")
model     = AutoModelForCausalLM.from_pretrained("eduhuemar001/tinyllama-german")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

#### Load dataset

In [5]:
streamed_dataset = load_dataset(
    "wikipedia",
    "20220301.de",
    split="train",
    streaming=True,
    trust_remote_code=True
)

dataset = list(islice(streamed_dataset, 900)) # Download 900 wikipedia articles
total_size = sum(sys.getsizeof(item["text"]) for item in dataset)
print(f"Approximate total size of Wikipedia texts: {total_size / 1024**2:.2f} MB")

README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

wikipedia.py:   0%|          | 0.00/36.7k [00:00<?, ?B/s]

Approximate total size of Wikipedia texts: 39.65 MB


## Preprocessing

#### Tokenize data

Each subword is being converted to a token with a byte pair encoder

In [6]:
tokens = []

for item in dataset:
    ids = tokenizer(item["text"], return_attention_mask=False, add_special_tokens=False).input_ids
    tokens.extend(ids + [tokenizer.eos_token_id])  # optional: add EOS after each article

Token indices sequence length is longer than the specified maximum sequence length for this model (4490 > 2048). Running this sequence through the model will result in indexing errors


#### Split tokens into chunks



Documents with a high amount of tokens exceed the maximum context window of 2048 tokens. The documents are split up into chunks of 512 tokens.

In [7]:
block_size = 512
total_length = len(tokens) - (len(tokens) % block_size)
tokens = tokens[:total_length] # Ensure the tokens array has a length of multiple of block size

chunks = [tokens[i:i + block_size] for i in range(0, total_length, block_size)] # Split tokens into 512-token chunks

#### Prepare chunks for training

In [8]:
class ChunkDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        ids = torch.tensor(self.chunks[idx], dtype=torch.long)
        return {
            "input_ids": ids,
            "labels": ids
        }

train_dataset = ChunkDataset(chunks)

## Set training arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./tinyllama-german-finetuned", # Where to save checkpoints
    per_device_train_batch_size=2, # 2 chunks are getting processed by GPU at the same time
    gradient_accumulation_steps=4, # Weights are being updated after 8 chunks are processed (2 chunks * 4 batches = 8 chunks)
    num_train_epochs=2,            # Model uses each chunk 2 times for training (dataset is processed 2 times iteratively)
    learning_rate=2e-4,            # Speed of learning (gradient step size)
    save_strategy="epoch",         # Save a model checkpoint after each epoch
    save_total_limit=2,            # Keep the 2 latest checkpoints
    logging_steps=10,              # Print loss/log info every 10 steps
    fp16=True,                     # Use mixed precision (faster/lower memory)
    report_to="none",              # No external logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

## Start training loop

#### Start initial training or resume training if checkpoint exists. The checkpoints are saved as long as colab session runs. Save model to huggingface after finished training.

In [10]:
checkpoint_dir = training_args.output_dir
resume_checkpoint = None

if os.path.isdir(checkpoint_dir):
    checkpoints = [
        os.path.join(checkpoint_dir, d)
        for d in os.listdir(checkpoint_dir)
        if d.startswith("checkpoint")
    ]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
        resume_checkpoint = latest

start_time = time.time()
if resume_checkpoint:
    print(f"Resuming from checkpoint: {resume_checkpoint}")
    trainer.train(resume_from_checkpoint=resume_checkpoint)
else:
    print("Starting fresh training...")
    trainer.train()
end_time = time.time()
print(f"Training took {end_time - start_time:.2f} seconds")

print("Training finished — pushing model to Hugging Face Hub...")
model.push_to_hub("eduhuemar001/tinyllama-german")
tokenizer.push_to_hub("eduhuemar001/tinyllama-german")

Starting fresh training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 32.12 MiB is free. Process 4612 has 14.71 GiB memory in use. Of the allocated memory 14.15 GiB is allocated by PyTorch, and 434.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Save data

#### Save to hugginface